In [1]:
!pip install -U dashscope

In [2]:
import cv2
import os
import math

def extract_frames(video_path, fps_target=7, output_dir="temp_frames", resize_factor=None):
    """
    Extract exactly fps_target frames per second by seeking to precise timestamps.
    Naming convention: <second>_<frame_in_second>.jpg
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = total_frames / video_fps

    print(f"Video info: {total_frames} frames, {video_fps:.2f} FPS, duration ≈ {duration_sec:.2f} sec")

    saved_count = 0
    max_full_second = math.floor(duration_sec)

    # Loop over each second
    for second_id in range(1, max_full_second + 1):
        for frame_in_second in range(1, fps_target + 1):
            # Target timestamp in seconds
            t = (second_id - 1) + (frame_in_second - 1) / fps_target
            if t > duration_sec:
                break

            # Seek to timestamp (milliseconds)
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
            ret, frame = cap.read()
            if not ret:
                continue

            # Resize if requested
            if resize_factor is not None and resize_factor > 0:
                new_w = int(frame.shape[1] * resize_factor)
                new_h = int(frame.shape[0] * resize_factor)
                frame = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

            filename = f"{second_id}_{frame_in_second}.jpg"
            filepath = os.path.join(output_dir, filename)
            cv2.imwrite(filepath, frame)
            saved_count += 1

            if saved_count % 100 == 0:
                print(f"Saved {saved_count} frames...")

    cap.release()

    expected_frames = max_full_second * fps_target
    print(f"✅ Extraction complete. Saved {saved_count} frames to '{output_dir}'")
    print(f"Expected ≈ {expected_frames} frames (plus possible partial second).")


In [3]:
video_file = '/home/ubuntu-user/Desktop/temp_video/videos/1B.avi'#'/media/ubuntu-user/KINGSTON/00Workspace_portable/videos/1B.avi'
#cp /media/ubuntu-user/KINGSTON/00Workspace_portable/videos/1B.avi /home/ubuntu-user/Desktop/temp_video/videos/1B.avi
fps_rate = 7
temp_folder = '/home/ubuntu-user/Desktop/portable/demo_temp_frames'

# Extract frames resized to 50% width & height
extract_frames(video_file, fps_target=fps_rate, output_dir=temp_folder, resize_factor=0.5)

Video info: 2196 frames, 7.00 FPS, duration ≈ 313.71 sec
Saved 100 frames...
Saved 200 frames...
Saved 300 frames...
Saved 400 frames...
Saved 500 frames...
Saved 600 frames...
Saved 700 frames...
Saved 800 frames...
Saved 900 frames...
Saved 1000 frames...
Saved 1100 frames...
Saved 1200 frames...
Saved 1300 frames...
Saved 1400 frames...
Saved 1500 frames...
Saved 1600 frames...
Saved 1700 frames...
Saved 1800 frames...
Saved 1900 frames...
Saved 2000 frames...
Saved 2100 frames...
✅ Extraction complete. Saved 2191 frames to '/home/ubuntu-user/Desktop/portable/demo_temp_frames'
Expected ≈ 2191 frames (plus possible partial second).


In [ ]:
#watch -n 1 nvidia-smi

In [22]:
assert os.getenv("DASHSCOPE_API_KEY"), "API Key 未设置！"

In [1]:
import os
from dashscope import Generation
import dashscope 

messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': '你是谁？'}
]
response = Generation.call(
    # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为：api_key = "sk-xxx",
    api_key=os.getenv("DASHSCOPE_API_KEY"), 
    model="qwen-plus",   # 模型列表：https://help.aliyun.com/model-studio/getting-started/models
    messages=messages,
    result_format="message"
)

if response.status_code == 200:
    print(response.output.choices[0].message.content)
else:
    print(f"HTTP返回码：{response.status_code}")
    print(f"错误码：{response.code}")
    print(f"错误信息：{response.message}")
    print("请参考文档：https://help.aliyun.com/model-studio/developer-reference/error-code")

你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊


In [2]:
print(response)

{"status_code": 200, "request_id": "e19c0d4c-8c30-9041-8848-229acd30881e", "code": "", "message": "", "output": {"text": null, "finish_reason": null, "choices": [{"finish_reason": "stop", "message": {"role": "assistant", "content": "你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊"}}]}, "usage": {"input_tokens": 22, "output_tokens": 66, "total_tokens": 88, "prompt_tokens_details": {"cached_tokens": 0}}}


In [5]:
import dashscope
import os

# 各地域配置不同，请根据实际地域修改
#dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/api/v1"

local_path = "/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/contraction_ub/1A_0152.mp4"
video_path = f"file://{local_path}"

system_prompt = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 

Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 

meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
warning: must response in english regardless of my location, timezone and system/browser language settings. 

When classifying stentor behaviors, keep in mind the following important points:
-) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
-) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
-) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos."""

messages=[
  {"role": "system", "content": system_prompt},
  {"role": "user", "content":[
            # fps 可参数控制视频抽帧频率，表示每隔 1/fps 秒抽取一帧，完整用法请参见：https://help.aliyun.com/zh/model-studio/use-qwen-by-calling-api?#2ed5ee7377fum
            {"video": video_path,"fps":7},
            {"text": "Analysis this video"}
        ]}
]


response = dashscope.MultiModalConversation.call(
    # 若没有配置环境变量， 请用百炼API Key将下行替换为： api_key ="sk-xxx"
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3-vl-235b-a22b-thinking',
    messages=messages
)

print(response.output.choices[0].message.content[0]["text"])



yes/95/At approximately 5.5 seconds, the stentor transitions from an extended trumpet shape to a condensed, shorter form, with a noticeable decrease in body length and a more rounded appearance. This meets the definition of a contraction as the organism changes from extended to condensed, even if not fully spherical.


In [6]:
print(response)


{"status_code": 200, "request_id": "fed7637a-f40b-929a-b176-ec30fbb1371a", "code": "", "message": "", "output": {"text": null, "finish_reason": null, "choices": [{"finish_reason": "stop", "message": {"role": "assistant", "content": [{"text": "yes/95/At approximately 5.5 seconds, the stentor transitions from an extended trumpet shape to a condensed, shorter form, with a noticeable decrease in body length and a more rounded appearance. This meets the definition of a contraction as the organism changes from extended to condensed, even if not fully spherical."}], "reasoning_content": "So, let's analyze the video. The stentor starts in an extended trumpet shape. Looking at the frames, around 5.5 seconds, the stentor's shape changes. It goes from the extended form to a more condensed, shorter shape. By 5.5 to 6.0 seconds, it's clearly contracted into a more ball-like or condensed form. The key is the change from extended to condensed. Even if it's not perfectly spherical, the shortening of t

In [48]:
import os
import glob
import json
import ollama

def process_video_clips(
    frame_dir,
    clip_length_sec=10,
    fps_target=7,
    output_json="/home/ubuntu-user/Desktop/portable/video_analysis.json",
    prompt=None,
    model="qwen3-vl:8b"
):
    """
    Process video frames in fixed-length clips and query LLM for each clip.
    Results are saved as JSON with clip ranges as keys and message content as values.
    """
    if prompt is None:
        prompt = "Describe what happens in these frames"

    # Load existing results safely
    results = {}
    if os.path.exists(output_json):
        try:
            with open(output_json, "r") as f:
                results = json.load(f)
        except (json.JSONDecodeError, ValueError):
            print(f"⚠ Warning: {output_json} is empty or invalid. Starting fresh.")
            results = {}

    # Collect all frames
    frames = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))

    # Group frames by second
    frames_by_second = {}
    for f in frames:
        fname = os.path.basename(f)
        second = int(fname.split("_")[0])  # e.g., "4_3.jpg" → second=4
        frames_by_second.setdefault(second, []).append(f)

    if not frames_by_second:
        print("⚠ No frames found in directory.")
        return

    max_second = max(frames_by_second.keys())

    # Iterate over clips
    for start_sec in range(1, max_second + 1, clip_length_sec):
        end_sec = min(start_sec + clip_length_sec - 1, max_second)
        clip_key = f"{start_sec}-{end_sec}"

        # Skip if already processed
        if clip_key in results:
            print(f"⏩ Skipping clip {clip_key}, already processed.")
            continue

        # Collect frames for this clip
        clip_frames = []
        for sec in range(start_sec, end_sec + 1):
            clip_frames.extend(frames_by_second.get(sec, []))

        if not clip_frames:
            continue

        # Build message
        messages = [
            {
                "role": "user",
                "content": prompt,
                "images": clip_frames
            }
        ]

        # Query Ollama
        print(f"▶ Processing clip {clip_key} with {len(clip_frames)} frames...")
        response = ollama.chat(model=model, messages=messages)

        # Save only the message content (JSON-serializable)
        results[clip_key] = response['message']['content']

        # Write JSON immediately
        with open(output_json, "w") as f:
            json.dump(results, f, indent=2)

        print(f"✅ Finished clip {clip_key}, saved to {output_json}")

    print("🎉 All clips processed.")


In [49]:
frame_dir = '/home/ubuntu-user/Desktop/portable/demo_temp_frames'
output_json = '/home/ubuntu-user/Desktop/portable/1B_video_analysis.json'

prompt = r"""You are good at identifying behaviors of stentors from video.
The uploaded video is about a stentor. The video consists of subsampled frames.
Tell me whether the stentor exhibits the "contraction" behavior, which means
that the Stentor changes from a "trumpet" (extended) shape to a "droplet"
(contracted) shape. Note that the contraction behavior can be quick, especially
in the subsampled video. You need to answer "yes" if the stentor exhibits
"contraction"; otherwise, "no". Moreover, you should explain when the
"contraction" happens."""

process_video_clips(
    frame_dir=frame_dir,
    clip_length_sec=10,
    fps_target=7,
    output_json=output_json,
    prompt=prompt,
    model="qwen3-vl:8b"
)


⚠ Warning: /home/ubuntu-user/Desktop/portable/1B_video_analysis.json is empty or invalid. Starting fresh.
▶ Processing clip 1-10 with 70 frames...
✅ Finished clip 1-10, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 11-20 with 70 frames...
✅ Finished clip 11-20, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 21-30 with 70 frames...
✅ Finished clip 21-30, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 31-40 with 70 frames...
✅ Finished clip 31-40, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 41-50 with 70 frames...
✅ Finished clip 41-50, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 51-60 with 70 frames...
✅ Finished clip 51-60, saved to /home/ubuntu-user/Desktop/portable/1B_video_analysis.json
▶ Processing clip 61-70 with 70 frames...
✅ Finished clip 61-70, saved to /home/ubuntu-user/Desktop/port

In [6]:
#system-role format -arc

import os
import shutil
import pandas as pd
import dashscope

# --- CONFIGURATION ---

DIR_base = '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg'
DIR_base = '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_pure' # only one behaviour in one video
DIR_base = '/Volumes/Green SSD/00Workspace_portable/bot_clips_test' # selected non segmented 

CSV_base = '/Users/kesiyun/Desktop/00workspace/bots/API_csvs'

labels = {
    'contraction': 'c',
    'alteration': 'a',
    'bending': 'b',
    'detachment': 'd'
}

DIR_list = [
    [
        os.path.join(DIR_base, k), 
        os.path.join(CSV_base, f'{v}_stentor_analysis_results.csv')
    ] 
    for k, v in labels.items()
]

for i in range (len(DIR_list)):
    INPUT_DIR = DIR_list[i][0]
    OUTPUT_CSV =DIR_list[i][1]
    #print(INPUT_DIR)
    #print(OUTPUT_CSV)


    MODEL_NAME = 'qwen3-vl-235b-a22b-thinking'
    #MODEL_NAME = 'qwen3-vl-32b-thinking'
    
    FPS_RATE = 7
    
    # --- PROMPT ---
    
    SYSTEM_PROMPT = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    
    Meanwhile please provide an integer contraction score for your judgement, where 0 = not a contraction and 100 = sure contraction.
    Response format:
    "[yes or no]/[contraction score]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """
    
    # --- MAIN PIPELINE ---
    
    # 1. Load processed records
    processed_files = set()
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing['filename'].tolist())
    
    # 2. List videos
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")
    print("model:", MODEL_NAME)
    print("dir:", INPUT_DIR)

    
    for video_name in all_videos:
        if video_name in processed_files:
            print(f"Skipping: {video_name}")
            continue
    
        video_path = os.path.join(INPUT_DIR, video_name)
        print(f"\nProcessing: {video_name}...")
    
        try:
            video_uri = f"file://{os.path.abspath(video_path)}"
    
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"video": video_uri, "fps": FPS_RATE},
                        {"text": "Analyze this video."}
                    ]
                }
            ]
    
            response = dashscope.MultiModalConversation.call(
                api_key=os.getenv('DASHSCOPE_API_KEY'),
                model=MODEL_NAME,
                messages=messages
            )
    
            # --- STATUS CHECK (NEW) ---
            if response.status_code != 200:
                print(f"❌ HTTP status: {response.status_code}")
                print(f"Error code: {getattr(response, 'code', 'N/A')}")
                print(f"Error message: {getattr(response, 'message', 'N/A')}")
                continue
    
            # --- PARSE RESPONSE ---
            ai_response = response.output.choices[0].message.content[0]["text"]
    
            # --- TOKEN USAGE PRINT (NEW, SAFE ACCESS) ---
            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)
                total_tokens = getattr(usage, "total_tokens", None)
                print(
                    f"✅ Token usage — "
                    f"input: {input_tokens}, output: {output_tokens}, total: {total_tokens}"
                )
            else:
                print("✅ Processed successfully (token usage not returned).")
    
            new_row = {
                'filename': video_name,
                'response': ai_response
            }
    
            # Append immediately (crash-safe)
            pd.DataFrame([new_row]).to_csv(
                OUTPUT_CSV,
                mode='a',
                index=False,
                header=not os.path.exists(OUTPUT_CSV)
            )
    
            print(f"Done: {video_name}")
    
        except Exception as e:
            print(f"❌ Exception processing {video_name}: {e}")
    
    print(f"\nAll done! Results saved to {OUTPUT_CSV}\n")

Found 77 videos. 0 already processed.
model: qwen3-vl-235b-a22b-thinking
dir: /Volumes/Green SSD/00Workspace_portable/bot_clips_test/contraction

Processing: 9A_0642_P.mp4...
✅ Token usage — input: 21853, output: 216, total: 22069
Done: 9A_0642_P.mp4

Processing: 4F_0038.mp4...
✅ Token usage — input: 21853, output: 238, total: 22091
Done: 4F_0038.mp4

Processing: 9A_1703_P.mp4...
✅ Token usage — input: 21853, output: 334, total: 22187
Done: 9A_1703_P.mp4

Processing: 9A_2011_A_P.mp4...
✅ Token usage — input: 21853, output: 163, total: 22016
Done: 9A_2011_A_P.mp4

Processing: 4D_1002.mp4...
✅ Token usage — input: 21853, output: 154, total: 22007
Done: 4D_1002.mp4

Processing: 1C_0550_A.mp4...
✅ Token usage — input: 21853, output: 151, total: 22004
Done: 1C_0550_A.mp4

Processing: 9A_0002_A_P.mp4...
✅ Token usage — input: 15077, output: 147, total: 15224
Done: 9A_0002_A_P.mp4

Processing: 15B_1048_A.mp4...
✅ Token usage — input: 21853, output: 147, total: 22000
Done: 15B_1048_A.mp4

Proc

KeyboardInterrupt: 

In [8]:
#multiple_arc
#system-role format

#https://billing-cost.console.aliyun.com/resource/spn/detail
import os
import shutil
import pandas as pd
import dashscope

# --- CONFIGURATION ---

DIR_base = '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_pure'
CSV_base = '/Users/kesiyun/Desktop/00workspace/bots/API_csvs'

labels = {
    'contraction': 'c',
    'alteration': 'a',
    'bending': 'b',
    'detachment': 'd',
}

DIR_list = [
    [
        os.path.join(DIR_base, k), 
        os.path.join(CSV_base, f'{v}_stentor_analysis_results.csv')
    ] 
    for k, v in labels.items()
]

for input_path, output_csv in DIR_list:
    INPUT_DIR = input_path
    OUTPUT_CSV = output_csv
    #print(INPUT_DIR)
    #print(OUTPUT_CSV)


    MODEL_NAME = 'qwen3-vl-235b-a22b-thinking'
    FPS_RATE = 7
    
    # --- PROMPT ---
    
    SYSTEM_PROMPT = r"""**Role:** You are an expert at identifying the behaviors of **Stentor**, a type of unicellular ciliated protist. 

    **Task:** The uploaded video shows a **Stentor** under magnification. Your goal is to identify which behaviors occur within this specific clip based on a 5-part taxonomy of common behaviors, which are defined below. 
    
    **Behavior Definitions:**
    - **(C) Contraction:** The **Stentor** changes from an extended trumpet shape to a more condensed or ball-like shape. This movement can be "Fast" (one frame) or "Slow" (multiple frames). Partial shortening in which the body length decreases noticeably but does not become a full sphere also counts as contraction.
    - **(B) Bending:** The **Stentor** noticeably curves or leans to one side while remaining attached to the surface and mostly or fully extended. Bending may occur in any direction. 
    - **(A) Ciliary Alteration:** A change in the beat or direction of the cilia (the hair-like structures around the “mouth” of the **Stentor**), which may shift the flow of surrounding particles. The cilia will often appear to “stand up straight” instead of pointing away from the body of the organism. 
    - **(D) Detachment:** The **Stentor** releases its holdfast and swims freely in the water. A free-swimming **Stentor** will usually be more compact in shape than a resting organism but less compact than a contracted one. 
    - **(R) Resting:** The **Stentor** remains fully extended at rest, and no transitions or specific avoidance behaviors are observed during the clip.
    
    **Instructions:**
    1. **Independent evaluation:** Evaluate this video independently. Do not use observations from any previous videos.
    2. **Focus:** Ignore any objects other than the **Stentor**, such as debris, algae, or glass needles.
    3. **Binary coding:** If multiple behaviors may be occurring, identify the most prominent one.
    
    **Response Format:**
    You must respond in English using the following structure:
    `[Behavior Code] / [Brief Reasoning]`
    
    - The **first part** is the single character (C, B, A, D, or R) for the most likely behavior.
    - The **second part** is your concise scientific reasoning for the annotation. 
    
    **Example Response:**
    `C/At 0.4 s the **Stentor** shows a rapid shortening of its body, which ends up in a spherical shape. While there is a slight curve (B) before the event, the contraction is the primary behavior. 

    """
    
    # --- MAIN PIPELINE ---
    
    # 1. Load processed records
    processed_files = set()
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing['filename'].tolist())
    
    # 2. List videos
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")
    
    for video_name in all_videos:
        if video_name in processed_files:
            print(f"Skipping: {video_name}")
            continue
    
        video_path = os.path.join(INPUT_DIR, video_name)
        print(f"\nProcessing: {video_name}...")
    
        try:
            video_uri = f"file://{os.path.abspath(video_path)}"
    
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"video": video_uri, "fps": FPS_RATE},
                        {"text": "Analyze this video."}
                    ]
                }
            ]
    
            response = dashscope.MultiModalConversation.call(
                api_key=os.getenv('DASHSCOPE_API_KEY'),
                model=MODEL_NAME,
                messages=messages
            )
    
            # --- STATUS CHECK (NEW) ---
            if response.status_code != 200:
                print(f"❌ HTTP status: {response.status_code}")
                print(f"Error code: {getattr(response, 'code', 'N/A')}")
                print(f"Error message: {getattr(response, 'message', 'N/A')}")
                continue
    
            # --- PARSE RESPONSE ---
            ai_response = response.output.choices[0].message.content[0]["text"]
    
            # --- TOKEN USAGE PRINT (NEW, SAFE ACCESS) ---
            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)
                total_tokens = getattr(usage, "total_tokens", None)
                print(
                    f"✅ Token usage — "
                    f"input: {input_tokens}, output: {output_tokens}, total: {total_tokens}"
                )
            else:
                print("✅ Processed successfully (token usage not returned).")
    
            new_row = {
                'filename': video_name,
                'response': ai_response
            }
    
            # Append immediately (crash-safe)
            pd.DataFrame([new_row]).to_csv(
                OUTPUT_CSV,
                mode='a',
                index=False,
                header=not os.path.exists(OUTPUT_CSV)
            )
    
            print(f"Done: {video_name}")
    
        except Exception as e:
            print(f"❌ Exception processing {video_name}: {e}")
    
    print(f"\nAll done! Results saved to {OUTPUT_CSV}")

Found 55 videos. 55 already processed.
Skipping: 9A_0642_P.mp4
Skipping: 4F_0038.mp4
Skipping: 9A_1703_P.mp4
Skipping: 4D_1002.mp4
Skipping: 17F_0328.mp4
Skipping: 11C_0036_P.mp4
Skipping: 3B_1239.mp4
Skipping: 15B_1243.mp4
Skipping: 10B_0032.mp4
Skipping: 15C_0019_P.mp4
Skipping: 10A_0205.mp4
Skipping: 1F_0121.mp4
Skipping: 9A_3714_P.mp4
Skipping: 1A_0152.mp4
Skipping: 3D_1402.mp4
Skipping: 17H_0151.mp4
Skipping: 6C_0009_P.mp4
Skipping: 10B_0008_P.mp4
Skipping: 14A_0045.mp4
Skipping: 1C_0930.mp4
Skipping: 13A_1328.mp4
Skipping: 9A_1051_P.mp4
Skipping: 8A_1611.mp4
Skipping: 10C_0012_P.mp4
Skipping: 8A_0353_P.mp4
Skipping: 9A_0838_P.mp4
Skipping: 9A_3330_P.mp4
Skipping: 11D_0105.mp4
Skipping: 13A_0206.mp4
Skipping: 13A_1050.mp4
Skipping: 15B_1602.mp4
Skipping: 12A_1355.mp4
Skipping: 15A_1244.mp4
Skipping: 9A_1343_P.mp4
Skipping: 6C_0512_P.mp4
Skipping: 17E_0111.mp4
Skipping: 3B_0415.mp4
Skipping: 1B_0325.mp4
Skipping: 11C_0358.mp4
Skipping: 1E_0247.mp4
Skipping: 15B_0018.mp4
Skipping: 1

In [3]:
#no move

#user only format -switchable
import os
import shutil
import pandas as pd
import dashscope

# --- CONFIGURATION ---


DIR_base = '/Volumes/Green SSD/00Workspace_portable/bot_clips'

CSV_base = '/Users/kesiyun/Desktop/00workspace/bots/API_csvs'

labels = {
    'selected_nomove': 'n'
}

DIR_list = [
    [
        os.path.join(DIR_base, k), 
        os.path.join(CSV_base, f'{v}_stentor_analysis_results.csv')
    ] 
    for k, v in labels.items()
]


for i in range (len(DIR_list)):
    INPUT_DIR = DIR_list[i][0]
    OUTPUT_CSV =DIR_list[i][1]
    #print(INPUT_DIR)
    #print(OUTPUT_CSV)


    MODEL_NAME = 'qwen3-vl-235b-a22b-thinking'
    #MODEL_NAME = 'qwen3-vl-235b-a22b-instruct'
    #MODEL_NAME = 'qwen3-vl-32b-thinking'

    ROLE_MODE = 'system-user'
    #ROLE_MODE = 'user-only' #'ollama'

    FPS_RATE = 7
    
    # --- PROMPT ---

    #ollama
    ollama_prompt_text = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 
    
    meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
    response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
    warning: must response in english regardless of my location, timezone and system/browser language settings. 
    
    When classifying stentor behaviors, keep in mind the following important points:
    -) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
    -) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
    -) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
    Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos."""

    API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    
    Meanwhile please provide an integer contraction score for your judgement, where 0 = not a contraction and 100 = sure contraction.
    Response format:
    "[yes or no]/[contraction score]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    no_score_API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    Response format:
    "[yes or no]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    

    #Ollama: user-only
    #prompt_text = ollama_prompt_text #original prompt used on ollama (ubuntu)
    prompt_text = API_system_prompt_text #to test the difference caused by modes
    
    #API_SYSTEM
    SYSTEM_PROMPT = no_score_API_system_prompt_text #API_system_prompt_text

    #API_SYSTEM
    USER_PROMPT =r"""Analyze this video."""
    
    # --- MAIN PIPELINE ---
    
    # 1. Load processed records
    processed_files = set()
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing['filename'].tolist())
    
    # 2. List videos
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")
    print("model:", MODEL_NAME)
    print("dir:", INPUT_DIR)
    print("mode:",ROLE_MODE)

    process_index=0
    
    for video_name in all_videos:
        if video_name in processed_files:
            print(f"Skipping: {video_name}")
            continue
    
        video_path = os.path.join(INPUT_DIR, video_name)
        print(f"\nProcessing: {video_name}...")
    
        try:
            video_uri = f"file://{os.path.abspath(video_path)}"

            if ROLE_MODE=='user-only': 
                messages = [{
                        "role": "user",
                        "content": [
                            {"text": prompt_text},
                            {"video": video_uri, "fps": FPS_RATE},
                        ]
                    }
                ]
    

            else: #ROLE_MODE=='system-user': 
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {"video": video_uri, "fps": FPS_RATE},
                            {"text": USER_PROMPT}
                        ]
                    }
                ]
        
            response = dashscope.MultiModalConversation.call(
                api_key=os.getenv('DASHSCOPE_API_KEY'),
                model=MODEL_NAME,
                messages=messages
            )
    
            # --- STATUS CHECK (NEW) ---
            if response.status_code != 200:
                print(f"❌ HTTP status: {response.status_code}")
                print(f"Error code: {getattr(response, 'code', 'N/A')}")
                print(f"Error message: {getattr(response, 'message', 'N/A')}")
                continue
    
            # --- PARSE RESPONSE ---
            ai_response = response.output.choices[0].message.content[0]["text"]
    
            # --- TOKEN USAGE PRINT (NEW, SAFE ACCESS) ---
            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)
                total_tokens = getattr(usage, "total_tokens", None)
                print(
                    f"✅ Token usage — "
                    f"input: {input_tokens}, output: {output_tokens}, total: {total_tokens}"
                )
            else:
                print("✅ Processed successfully (token usage not returned).")
    
            new_row = {
                'filename': video_name,
                'response': ai_response
            }
    
            # Append immediately (crash-safe)
            pd.DataFrame([new_row]).to_csv(
                OUTPUT_CSV,
                mode='a',
                index=False,
                header=not os.path.exists(OUTPUT_CSV)
            )
    
            print(f"Done: {video_name}, {process_index}/{len(all_videos)}")
    
        except Exception as e:
            print(f"❌ Exception processing {video_name}: {e}")
    
    print(f"\nAll done! Results saved to {OUTPUT_CSV}\n")

Found 85 videos. 85 already processed.
model: qwen3-vl-235b-a22b-thinking
dir: /Volumes/Green SSD/00Workspace_portable/bot_clips/selected_nomove
mode: system-user
Skipping: 15D_0505.mp4
Skipping: 3B_2325.mp4
Skipping: 3C_0255.mp4
Skipping: 9A_1355.mp4
Skipping: 9A_0505.mp4
Skipping: 18B_4505.mp4
Skipping: 11A_0005.mp4
Skipping: 17D_0025.mp4
Skipping: 6A_0145.mp4
Skipping: 8B_0525.mp4
Skipping: 1A_0255.mp4
Skipping: 4B_0235.mp4
Skipping: 13A_0915.mp4
Skipping: 8B_2325.mp4
Skipping: 1F_0225.mp4
Skipping: 14D_1035.mp4
Skipping: 14A_0645.mp4
Skipping: 15C_0305.mp4
Skipping: 5A_0115.mp4
Skipping: 17C_0025.mp4
Skipping: 4D_0155.mp4
Skipping: 13A_2425.mp4
Skipping: 17F_0005.mp4
Skipping: 17D_0645.mp4
Skipping: 18A_0025.mp4
Skipping: 2A_0215.mp4
Skipping: 1C_0845.mp4
Skipping: 1E_0405.mp4
Skipping: 13A_0035.mp4
Skipping: 14C_1255.mp4
Skipping: 8A_1345.mp4
Skipping: 8B_1855.mp4
Skipping: 9A_3055.mp4
Skipping: 18B_1035.mp4
Skipping: 17E_0045.mp4
Skipping: 11C_0345.mp4
Skipping: 10C_0905.mp4
Skip

In [5]:
#active portal

#test: using strictly same prompt to ollama
#user only format -switchable
import os
import shutil
import pandas as pd
import dashscope

# --- CONFIGURATION ---

DIR_base = '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg'
#DIR_base = '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_pure' # only one behaviour in one video
#DIR_base = '/Volumes/Green SSD/00Workspace_portable/bot_clips_test' # selected non segmented 
#DIR_base = '/Volumes/Green SSD/00Workspace_portable/bot_clips_testset'

CSV_base = '/Users/kesiyun/Desktop/00workspace/bots/API_csvs'

labels = {
    'contraction': 'c',
    'alteration': 'a',
    'bending': 'b',
    'detachment': 'd'
}

DIR_list = [
    [
        os.path.join(DIR_base, k), 
        os.path.join(CSV_base, f'{v}_stentor_analysis_results.csv')
    ] 
    for k, v in labels.items()
]


for i in range (len(DIR_list)):
    INPUT_DIR = DIR_list[i][0]
    OUTPUT_CSV =DIR_list[i][1]
    #print(INPUT_DIR)
    #print(OUTPUT_CSV)


    MODEL_NAME = 'qwen3-vl-235b-a22b-thinking'
    #MODEL_NAME = 'qwen3-vl-235b-a22b-instruct'
    #MODEL_NAME = 'qwen3-vl-32b-thinking'

    ROLE_MODE = 'system-user'
    #ROLE_MODE = 'user-only' #'ollama'

    FPS_RATE = 7
    
    # --- PROMPT ---

    #ollama
    ollama_prompt_text = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 
    
    meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
    response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
    warning: must response in english regardless of my location, timezone and system/browser language settings. 
    
    When classifying stentor behaviors, keep in mind the following important points:
    -) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
    -) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
    -) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
    Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos."""

    API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    
    Meanwhile please provide an integer contraction score for your judgement, where 0 = not a contraction and 100 = sure contraction.
    Response format:
    "[yes or no]/[contraction score]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    no_score_API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    Response format:
    "[yes or no]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    

    #Ollama: user-only
    #prompt_text = ollama_prompt_text #original prompt used on ollama (ubuntu)
    prompt_text = API_system_prompt_text #to test the difference caused by modes
    
    #API_SYSTEM
    SYSTEM_PROMPT = no_score_API_system_prompt_text #API_system_prompt_text

    #API_SYSTEM
    USER_PROMPT =r"""Analyze this video."""
    
    # --- MAIN PIPELINE ---
    
    # 1. Load processed records
    processed_files = set()
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing['filename'].tolist())
    
    # 2. List videos
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")
    print("model:", MODEL_NAME)
    print("dir:", INPUT_DIR)
    print("mode:",ROLE_MODE)

    process_index=0
    
    for video_name in all_videos:
        if video_name in processed_files:
            print(f"Skipping: {video_name}")
            continue
    
        video_path = os.path.join(INPUT_DIR, video_name)
        print(f"\nProcessing: {video_name}...")
    
        try:
            video_uri = f"file://{os.path.abspath(video_path)}"

            if ROLE_MODE=='user-only': 
                messages = [{
                        "role": "user",
                        "content": [
                            {"text": prompt_text},
                            {"video": video_uri, "fps": FPS_RATE},
                        ]
                    }
                ]
    

            else: #ROLE_MODE=='system-user': 
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {"video": video_uri, "fps": FPS_RATE},
                            {"text": USER_PROMPT}
                        ]
                    }
                ]
        
            response = dashscope.MultiModalConversation.call(
                api_key=os.getenv('DASHSCOPE_API_KEY'),
                model=MODEL_NAME,
                messages=messages
            )
    
            # --- STATUS CHECK (NEW) ---
            if response.status_code != 200:
                print(f"❌ HTTP status: {response.status_code}")
                print(f"Error code: {getattr(response, 'code', 'N/A')}")
                print(f"Error message: {getattr(response, 'message', 'N/A')}")
                continue
    
            # --- PARSE RESPONSE ---
            ai_response = response.output.choices[0].message.content[0]["text"]
    
            # --- TOKEN USAGE PRINT (NEW, SAFE ACCESS) ---
            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)
                total_tokens = getattr(usage, "total_tokens", None)
                print(
                    f"✅ Token usage — "
                    f"input: {input_tokens}, output: {output_tokens}, total: {total_tokens}"
                )
            else:
                print("✅ Processed successfully (token usage not returned).")
    
            new_row = {
                'filename': video_name,
                'response': ai_response
            }
    
            # Append immediately (crash-safe)
            pd.DataFrame([new_row]).to_csv(
                OUTPUT_CSV,
                mode='a',
                index=False,
                header=not os.path.exists(OUTPUT_CSV)
            )
    
            print(f"Done: {video_name}, {process_index}/{len(all_videos)}")
    
        except Exception as e:
            print(f"❌ Exception processing {video_name}: {e}")
    
    print(f"\nAll done! Results saved to {OUTPUT_CSV}\n")

Found 77 videos. 77 already processed.
model: qwen3-vl-235b-a22b-instruct
dir: /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/contraction
mode: system-user
Skipping: 9A_0642_P.mp4
Skipping: 4F_0038.mp4
Skipping: 9A_1703_P.mp4
Skipping: 9A_2011_A_P.mp4
Skipping: 4D_1002.mp4
Skipping: 1C_0550_A.mp4
Skipping: 9A_0002_A_P.mp4
Skipping: 15B_1048_A.mp4
Skipping: 13A_1730_A.mp4
Skipping: 17G_0038_A.mp4
Skipping: 17F_0328.mp4
Skipping: 11C_0036_P.mp4
Skipping: 3B_1239.mp4
Skipping: 15B_1243.mp4
Skipping: 10B_0032.mp4
Skipping: 9A_2655_A_P.mp4
Skipping: 9A_2952_A_P.mp4
Skipping: 15C_0019_P.mp4
Skipping: 10A_0205.mp4
Skipping: 1F_0121.mp4
Skipping: 9A_3714_P.mp4
Skipping: 1A_0152.mp4
Skipping: 3D_1402.mp4
Skipping: 17H_0151.mp4
Skipping: 15B_0210_A_P.mp4
Skipping: 6C_0009_P.mp4
Skipping: 3B_0047_A.mp4
Skipping: 10B_0008_P.mp4
Skipping: 17D_0011_A.mp4
Skipping: 14A_0045.mp4
Skipping: 1C_0930.mp4
Skipping: 13A_1328.mp4
Skipping: 9A_1051_P.mp4
Skipping: 8A_1611.mp4
Skipping: